# Quantized Test

In [10]:
import numpy as np
from tensorflow import keras
import tensorflow as tf

def convert_data(x_data: np.ndarray, y_data: np.ndarray):
    # hacky way to increase the size of a (unknown)x28x28 to (unknown)x32x32
    # by appending two zeros at all sides of the 28x28 matrix
    tmp = np.zeros((x_data.shape[0], 2, x_data.shape[2]))
    x_data = np.append(x_data, tmp, axis=1)
    x_data = np.append(tmp, x_data, axis=1)
    tmp = np.zeros((x_data.shape[0], x_data.shape[1], 2))
    x_data = np.append(x_data, tmp, axis=2)
    x_data = np.append(tmp, x_data, axis=2)

    # change dimensions: (unknown)x32x32 to (unknown)x32x32x1
    x_data = np.expand_dims(x_data, -1)
    # change dimensions: (unknown) to (unknown)x10
    y_data = keras.utils.to_categorical(y_data, 10)

    return (x_data - 128, y_data)


# for now we are just interested in the training data and not the test data
(X_train, y_train), (_, _) = tf.keras.datasets.mnist.load_data()

(X_train, y_train) = convert_data(X_train, y_train)


In [11]:
import sys

np.set_printoptions(threshold=sys.maxsize)

#images = (X_train - 128).astype(np.int8)
images = X_train.astype(np.int8)
#images = images.view(np.int8)
#images = X_train / 255

images[0]

array([[[-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128]],

       [[-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
        [-128],
      

### Load model

In [12]:
import numpy as np
import tensorflow as tf

# Load TFLite model and allocate tensors.
interpreter = tf.lite.Interpreter(model_path="export/quantized_model.tflite")
interpreter.allocate_tensors()

# Get input and output tensors.
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Test model on random input data.
input_shape = input_details[0]['shape']

In [14]:
succ = 0
for i in range(len(images)):
    interpreter.set_tensor(input_details[0]['index'], np.array([images[i]], dtype=np.float32))
    #interpreter.set_tensor(input_details[0]['index'], tf.cast(np.random.uniform(-128, 127, input_shape), tf.int8))
    #print(tf.cast(np.random.uniform(-128, 127, input_shape), tf.int8))
    #print(tf.cast(np.array([images[0]], dtype=np.int8), tf.int8))
    interpreter.invoke()

    # The function `get_tensor()` returns a copy of the tensor data.
    # Use `tensor()` in order to get a pointer to the tensor.
    # print(y_train[9])

    output_data = interpreter.get_tensor(output_details[0]['index'])
    if output_data.argmax() == y_train[i].argmax():
        succ += 1

print(succ / len(images))

0.9748333333333333
